In [ ]:
import pandas as pd
import torch
print("GPU available:", torch.cuda.is_available())

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

#filter data 
langlst = ['ko','ar','te']

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]

# df_train_filtered


## Fine-tune one multilingual Transformer QA model

In [ ]:
from datasets import Dataset
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)
torch.manual_seed(42)

train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)

def prepare_examples(examples):
    tokens = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", 
        max_length=384, 
        stride=128,
        return_overflowing_tokens=True, 
        return_offsets_mapping=True,
        padding="max_length",
    )
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    offsets = tokens.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, example_number in enumerate(example_numbers):
        cls_position = tokens["input_ids"][i].index(tokenizer.cls_token_id)
        start_position = cls_position
        end_position = cls_position

        if examples["answerable"][example_number]:
            answer_start = int(examples["answer_start"][example_number])
            answer_end = answer_start + len(examples["answer"][example_number])
            context_tokens = [
                j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
            ]
            if (context_tokens
                and offsets[i][context_tokens[0]][0] <= answer_start
                and offsets[i][context_tokens[-1]][1] >= answer_end):
                answer_tokens = [
                    j for j in context_tokens
                    if offsets[i][j][0] < answer_end
                    and offsets[i][j][1] > answer_start
                ]
                if answer_tokens:
                    start_position = answer_tokens[0]
                    end_position = answer_tokens[-1]

        start_positions.append(start_position)
        end_positions.append(end_position)

    tokens["start_positions"] = start_positions
    tokens["end_positions"] = end_positions
    return tokens

train_features = train_examples.map(
    prepare_examples, batched=True,
    remove_columns=train_examples.column_names,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-multilingual-cased"
)

training_args = TrainingArguments(
    output_dir="multilingual-qa", 
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    fp16=True,
    learning_rate=3e-5,
    seed=42, save_strategy="no", report_to="none",
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_features)
trainer.train()

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.bias                            | MISSING    | 
qa_outputs.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("multilingual-qa")
tokenizer.save_pretrained("multilingual-qa")

In [ ]:
import numpy as np

validation_examples = Dataset.from_pandas(
    df_validation_filtered, preserve_index=False
)

def prepare_validation(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    # Local example indices within this preprocessing batch.
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    tokens["example_index"] = example_numbers

    for i in range(len(tokens["input_ids"])):
        sequence_ids = tokens.sequence_ids(i)
        # Only context tokens can belong to an extracted answer.
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    return tokens

# One batch makes example_index refer to the whole validation dataset.
validation_features = validation_examples.map(
    prepare_validation,
    batched=True,
    batch_size=len(validation_examples),
    remove_columns=validation_examples.column_names,
)

# Metadata is needed for decoding, but must not be passed to the model.
model_inputs = validation_features.remove_columns(
    ["offset_mapping", "example_index"]
)
start_logits, end_logits = trainer.predict(model_inputs).predictions

n = len(validation_examples)
null_scores = np.full(n, np.inf)
span_scores = np.full(n, -np.inf)
best_answers = [""] * n

# A fixed decoding choice: number of WordPiece tokens, not words.
MAX_ANSWER_LENGTH = 30

for feature_index, feature in enumerate(validation_features):
    example_index = feature["example_index"]
    offsets = feature["offset_mapping"]
    starts = start_logits[feature_index]
    ends = end_logits[feature_index]

    cls_index = feature["input_ids"].index(tokenizer.cls_token_id)
    null_scores[example_index] = min(
        null_scores[example_index],
        float(starts[cls_index] + ends[cls_index]),
    )

    # Search all valid context spans up to MAX_ANSWER_LENGTH.
    for start, start_offset in enumerate(offsets):
        if start_offset is None or start_offset[1] <= start_offset[0]:
            continue

        for end in range(
            start, min(start + MAX_ANSWER_LENGTH, len(offsets))
        ):
            end_offset = offsets[end]
            if end_offset is None:
                break
            if end_offset[1] <= end_offset[0]:
                continue

            score = float(starts[start] + ends[end])
            if score > span_scores[example_index]:
                span_scores[example_index] = score
                context = validation_examples[example_index]["context"]
                best_answers[example_index] = context[
                    start_offset[0]:end_offset[1]
                ]

assert np.isfinite(null_scores).all()
assert np.isfinite(span_scores).all()

no_answer_scores = null_scores - span_scores

In [ ]:
from sklearn.metrics import f1_score

scores = np.asarray(no_answer_scores)
gold_answerable = df_validation_filtered["answerable"].to_numpy(dtype=bool)

thresholds = []

# First, test a cutoff that predicts no answer for everyone.
threshold_below_all_scores = scores.min() - 1
thresholds.append(threshold_below_all_scores)

# np.unique returns the distinct scores in ascending order.
unique_scores = np.unique(scores)

# Then test each score as a cutoff.
for score in unique_scores:
    thresholds.append(score)

best_threshold = thresholds[0]
best_macro_f1 = -1

for threshold in thresholds:
    predicted_answerable = scores <= threshold

    macro_f1 = f1_score(
        gold_answerable,
        predicted_answerable,
        labels=[False, True],
        average="macro",
        zero_division=0,
    )

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

NO_ANSWER_THRESHOLD = best_threshold

print("Threshold:", NO_ANSWER_THRESHOLD)
print("Validation macro-F1:", best_macro_f1)

In [ ]:
predicted_answers = [
    answer if score <= NO_ANSWER_THRESHOLD else ""
    for answer, score in zip(best_answers, scores)
]

## Evaluation (Section 1)

In [ ]:
from evaluate import load as load_metric
from sklearn.metrics import classification_report

# Lab 6 recommends squad_v2 for the project's unanswerable examples.
squad_metric = load_metric("squad_v2")

evaluation = df_validation_filtered.copy()
evaluation["prediction"] = predicted_answers
evaluation["pred_answerable"] = evaluation["prediction"] != ""

results = []
for language in ["overall", "ar", "ko", "te"]:
    group = evaluation if language == "overall" else evaluation[evaluation["lang"] == language]
    formatted_predictions = []
    gold = []

    for example_id, example in group.iterrows():
        formatted_predictions.append({
            "id": str(example_id),
            "prediction_text": example["prediction"],
            # Required field: encode the already-made abstention decision.
            # This is a binary indicator, not a calibrated probability.
            "no_answer_probability": float(not example["pred_answerable"]),
        })
        gold.append({
            "id": str(example_id),
            "answers": {
                "text": [example["answer"]] if example["answerable"] else [],
                "answer_start": [int(example["answer_start"])] if example["answerable"] else [],
            },
        })

    squad_results = squad_metric.compute(
        references=gold, predictions=formatted_predictions
    )
    report = classification_report(
        group["answerable"], group["pred_answerable"],
        labels=[False, True], target_names=["unanswerable", "answerable"],
        output_dict=True, zero_division=0,
    )

    row = {
        "language": language,
        "N": len(group),
        "SQuAD_v2_EM": squad_results["exact"],
        "SQuAD_v2_token_F1": squad_results["f1"],
        "answerability_macro_F1": 100 * report["macro avg"]["f1-score"],
    }
    for label in ["unanswerable", "answerable"]:
        row[f"{label}_P"] = 100 * report[label]["precision"]
        row[f"{label}_R"] = 100 * report[label]["recall"]
        row[f"{label}_F1"] = 100 * report[label]["f1-score"]
    results.append(row)

results_table = pd.DataFrame(results)
display(results_table.round(2))